# CST 407 PYTHON FOR AI
### Homework 2



### Import needed libraries


In [28]:
import os
import shutil
import tensorflow as tf
import keras
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras import layers, models
from keras import optimizers

### 1. After downloading and uncompressing it, create a new dataset called "SmallDataset" containing three subsets: a training set with 2,000 unique samples of each class, a validation set with 1000 unique samples of each class, and a test set with 10000 unique samples of each class (all data should come from train data set of the original data set)

In [24]:
original_dataset_dir = '/Users/yaelroque/Downloads/CST407/Homework 2/Dataset/train'
base_dir = '/Users/yaelroque/Downloads/CST407/Homework 2/SmallDataset'

train_dir = os.path.join(base_dir, 'train')
validation_dir = os.path.join(base_dir, 'validation')
test_dir = os.path.join(base_dir, 'test')

train_cats_dir = os.path.join(train_dir, 'cats')
train_dogs_dir = os.path.join(train_dir, 'dogs')

validation_cats_dir = os.path.join(validation_dir, 'cats')
validation_dogs_dir = os.path.join(validation_dir, 'dogs')

test_cats_dir = os.path.join(test_dir, 'cats')
test_dogs_dir = os.path.join(test_dir, 'dogs')

for d in [base_dir, train_dir, validation_dir, test_dir,
          train_cats_dir, train_dogs_dir,
          validation_cats_dir, validation_dogs_dir,
          test_cats_dir, test_dogs_dir]:
    os.makedirs(d, exist_ok=True)

copy_plan = [
    ('cat', range(0, 2000), train_cats_dir),
    ('cat', range(2000, 3000), validation_cats_dir),
    ('cat', range(3000, 4000), test_cats_dir),

    ('dog', range(0, 2000), train_dogs_dir),
    ('dog', range(2000, 3000), validation_dogs_dir),
    ('dog', range(3000, 4000), test_dogs_dir),
]

for animal, indices, dst_dir in copy_plan:
    for i in indices:
        fname = f'{animal}.{i}.jpg'
        shutil.copyfile(os.path.join(original_dataset_dir, fname),os.path.join(dst_dir, fname))

### 2. Verify dimension of your data set by printing out number of each images on each directory

In [25]:
print('Total training cat images:', len(os.listdir(train_cats_dir)))
print('Total validation cat images:', len(os.listdir(validation_cats_dir)))
print('Total test cat images:', len(os.listdir(test_cats_dir)))

print('Total training dog images:', len(os.listdir(train_dogs_dir)))
print('Total validation dog images:', len(os.listdir(validation_dogs_dir)))
print('Total test dog images:', len(os.listdir(test_dogs_dir)))

Total training cat images: 2000
Total validation cat images: 1000
Total test cat images: 1000
Total training dog images: 2000
Total validation dog images: 1000
Total test dog images: 1000


### 3. Preprocess your images
 - Make sure images are converted to 150x150x3
 - Make sure image pixels are in float and their magnitudes are between 0 and 1

In [26]:
train_datagen      = ImageDataGenerator(rescale=1./255)
validation_datagen = ImageDataGenerator(rescale=1./255)
test_datagen       = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150, 150),
    batch_size=20,
    class_mode='binary')

validation_generator = validation_datagen.flow_from_directory(
    validation_dir,
    target_size=(150, 150),
    batch_size=20,
    class_mode='binary')

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(150, 150),
    batch_size=20,
    class_mode='binary')

images, labels = next(train_generator)
print('Batch shape:', images.shape) 
print('Data type:',   images.dtype)       
print('Min pixel:',   images.min())        
print('Max pixel:',   images.max())   


Found 4000 images belonging to 2 classes.
Found 2000 images belonging to 2 classes.
Found 2000 images belonging to 2 classes.
Batch shape: (20, 150, 150, 3)
Data type: float32
Min pixel: 0.0
Max pixel: 1.0


### 4. Construct your first NN model: Use 5 conv layers and one dense layer before the output layer.


In [27]:
model = models.Sequential()

model.add(layers.Conv2D(32,  (3, 3), activation='relu', input_shape=(150, 150, 3)))
model.add(layers.MaxPooling2D((2, 2)))

model.add(layers.Conv2D(64,  (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))

model.add(layers.Conv2D(128, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))

model.add(layers.Conv2D(128, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))

model.add(layers.Conv2D(256, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))

model.add(layers.Flatten())

model.add(layers.Dense(512, activation='relu'))

model.add(layers.Dense(1, activation='sigmoid'))

model.summary()

/Users/yaelroque/anaconda3/envs/YAEL/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 148, 148, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 74, 74, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 72, 72, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 36, 36, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 34, 34, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 17, 17, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 15, 15, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 7, 7, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 5, 5, 256)      │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 2, 2, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       524,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           513 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,061,313 (4.05 MB)

 Trainable params: 1,061,313 (4.05 MB)

 Non-trainable params: 0 (0.00 B)

### 5. Choose appropriate loss function, optimizer and metrics.

In [29]:
model.compile(
    loss='binary_crossentropy',
    optimizer=optimizers.RMSprop(learning_rate=1e-4),
    metrics=['accuracy']
)

### 6. Choose appropriate values for *steps_per_epoch*, *epochs* and *validation_steps*
### 7. Run your model

In [30]:
history = model.fit(
    train_generator,
    steps_per_epoch=200,
    epochs=30,
    validation_data=validation_generator,
    validation_steps=100
)

Epoch 1/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 43s 208ms/step - accuracy: 0.5290 - loss: 0.6889 - val_accuracy: 0.5910 - val_loss: 0.6776
Epoch 2/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 44s 220ms/step - accuracy: 0.5968 - loss: 0.6691 - val_accuracy: 0.6110 - val_loss: 0.6575
Epoch 3/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 65s 324ms/step - accuracy: 0.6398 - loss: 0.6360 - val_accuracy: 0.6650 - val_loss: 0.6132
Epoch 4/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 64s 321ms/step - accuracy: 0.6705 - loss: 0.5991 - val_accuracy: 0.6740 - val_loss: 0.5956
Epoch 5/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 65s 326ms/step - accuracy: 0.6927 - loss: 0.5819 - val_accuracy: 0.6725 - val_loss: 0.5905
Epoch 6/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 66s 331ms/step - accuracy: 0.7107 - loss: 0.5587 - val_accuracy: 0.6925 - val_loss: 0.5682
Epoch 7/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 62s 310ms/step - accuracy: 0.7265 - loss: 0.5404 - val_accuracy: 0.7090 - val_loss: 0.5534
Epoch 8/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 63s 314ms/step - accuracy: 0.7523 - loss: 0